# Multilogue-native soft PPCA + document distance

**Purpose.** Recreate the soft-membership / distance *Address* slice of the public-dialogue pipeline by calling **multilogue only** — no `pub_dialogue.address` soft/distance bodies. Uses frozen checkpoints under `checkpoints/` and tables under `outputs/`.

**Also.** Codifies the 2026-10-16 gap-analysis decisions (what multilogue owns vs what stays study-local) as executable checks.

**Not in scope here.** LLM extraction, clustering, Part R1–R3 ℓ definitions, full multiverse artefact loading. Those stay in pub notebooks / `pub_dialogue`.

**Branch.** `cip0005-depend-multilogue` — confirmation notebook before merging the adopt PR.


In [ ]:
# Setup: local multilogue checkout preferred over the git URL pin.
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

ROOT = Path.cwd()
if not (ROOT / "checkpoints").is_dir():
    # Allow running from notebooks/
    if (ROOT.parent / "checkpoints").is_dir():
        ROOT = ROOT.parent
assert (ROOT / "checkpoints").is_dir(), f"checkpoints/ not found from {Path.cwd()}"

# Prefer sibling multilogue checkout for unreleased CIP-0006/0007 APIs.
for cand in (ROOT.parent / "multilogue", Path.home() / "mlatcl" / "multilogue"):
    if (cand / "multilogue" / "soft.py").is_file():
        sys.path.insert(0, str(cand))
        print("Using multilogue from", cand)
        break

from multilogue import soft as ms
from multilogue import distance as dd
from multilogue.weight import soft_doc_profiles, soft_group_compare

print("multilogue soft SIGMA2_RULES:", list(ms.SIGMA2_RULES))
print("ROOT =", ROOT)


## Gap-analysis decisions (2026-10-16)

| Priority | Decision | Owner | Status in multilogue |
|----------|----------|-------|----------------------|
| P0 | Soft PPCA + σ²-by-EMD + document EMD live in multilogue; consumers thin | CIP-0005 | Merged |
| P1 | σ² neighbour rules (`one_step_harder/softer`, `hard`) + `SIGMA2_RULES` | CIP-0006 | Merged |
| P2 | Soft-weighted doc profiles + group compare | CIP-0007 | Merged |
| P3 | StageRunner / LiteLLM client adapter | keep optional / local | Not required for this notebook |
| — | Study ℓ (R1–R3), tech filters, PDF access, legacy σ² (`min_emd`, `median`) | **keep local** | — |
| — | Formats `hard` → `None` (true one-hot), not min-σ² PPCA | **keep local** (formats adapter) | — |
| — | `checkpoints/soft_membership.npy` = phrase×cluster **cosine**, not PPCA | different object; do not confuse | — |

The cells below assert the promote/keep split where we can do so offline.


In [ ]:
# --- Gap checklist (structural; no data yet)
checks = []

def check(name: str, ok: bool, detail: str = ""):
    checks.append({"check": name, "pass": bool(ok), "detail": detail})
    mark = "PASS" if ok else "FAIL"
    print(f"[{mark}] {name}" + (f" — {detail}" if detail else ""))

check(
    "P1 neighbour rules exported",
    all(k in ms.SIGMA2_RULES for k in ("constrained", "one_step_harder", "one_step_softer", "hard")),
    str(list(ms.SIGMA2_RULES)),
)
check("P2 soft_doc_profiles callable", callable(soft_doc_profiles))
check("P2 soft_group_compare callable", callable(soft_group_compare))
check("P0 compute_mixture_weights callable", callable(ms.compute_mixture_weights))
check("P0 sigma2_sweep_select callable", callable(ms.sigma2_sweep_select))
check("P0 document distance entrypoint", callable(dd.run_document_distance_analysis))

# Keep-local: legacy rules must NOT be in multilogue.SIGMA2_RULES
check(
    "legacy min_emd/median stay out of multilogue.SIGMA2_RULES",
    "min_emd" not in ms.SIGMA2_RULES and "median" not in ms.SIGMA2_RULES,
)


## Load frozen artefacts

- `checkpoints/cluster_centroids.npy` — concern cluster centroids  
- `outputs/framing_lens_mappings.json` — canonical lens → cluster_ids  
- `outputs/extracted_concerns.csv` — phrases with `cluster_id`, `technology_meta`  
- `outputs/ai_vs_nonai_lens_doc_weighted.csv` — saved soft doc-weighted AI−nonAI gaps (headline table)

Per-run lens maps (`framing_lens_mappings_run_*.json`) are **not** in this tree. For σ² stability we build **synthetic** alternate mappings by perturbing the canonical map (methodology confirmation only — not a claim that we recover the paper's published σ²*).


In [ ]:
CKPT = ROOT / "checkpoints"
OUT = ROOT / "outputs"

centroids = np.load(CKPT / "cluster_centroids.npy")
mapping = json.loads((OUT / "framing_lens_mappings.json").read_text())
phrases = pd.read_csv(OUT / "extracted_concerns.csv")
saved_gaps = pd.read_csv(OUT / "ai_vs_nonai_lens_doc_weighted.csv")

# Phrase×cluster cosine soft_membership is a different object from PPCA mixture weights.
cosine_sm = np.load(CKPT / "soft_membership.npy")
check(
    "cosine soft_membership is phrase×cluster (not PPCA lens W)",
    cosine_sm.shape[0] == len(phrases) and cosine_sm.shape[1] == centroids.shape[0],
    f"shape={cosine_sm.shape}; n_phrases={len(phrases)}; k={centroids.shape[0]}",
)

print("centroids", centroids.shape)
print("n_lenses", len(mapping), "lenses:", list(mapping))
print("phrases", phrases.shape, "AI share", (phrases.technology_meta == "AI").mean().round(3))
print("saved gap table:\n", saved_gaps.to_string(index=False))


## Soft mixture weights (CIP-0005) → recreate AI vs non-AI doc-weighted gaps

Recompute PPCA mixture weights with multilogue, then phrase-mean AI vs non-AI shares (document-weighted baseline in the sense of the saved table: mean over non-AI phrases). Compare to `ai_vs_nonai_lens_doc_weighted.csv` by **Spearman rank** and **sign agreement** (exact σ² used for the saved table is not recorded here; we scan a grid and take the best rank match).

**Note.** Absolute share levels need not match bit-for-bit without the original `mixture_weights.npy` artefact (missing from this tree). Rank/sign agreement is the confirmation criterion for “same scientific claim about which lenses AI over/under-indexes.”


In [ ]:
def doc_weighted_ai_minus_nonai(W: np.ndarray, names: list[str], phrases: pd.DataFrame) -> pd.Series:
    valid = (phrases.cluster_id >= 0) & (phrases.cluster_id < W.shape[0])
    pw = W[phrases.loc[valid, "cluster_id"].to_numpy(dtype=int)]
    tech = phrases.loc[valid, "technology_meta"].to_numpy()
    ai = tech == "AI"
    diff = pw[ai].mean(0) - pw[~ai].mean(0)
    return pd.Series(diff, index=names)


grid = np.logspace(-4, 0, 25)
rows = []
for s2 in grid:
    W, names = ms.compute_mixture_weights(centroids, mapping, float(s2))
    ser = doc_weighted_ai_minus_nonai(W, names, phrases)
    aligned = ser.reindex(saved_gaps.lens).to_numpy()
    target = saved_gaps.ai_minus_doc_weighted.to_numpy()
    rho = float(spearmanr(aligned, target).correlation)
    mae = float(np.mean(np.abs(aligned - target)))
    sign_agree = float(np.mean(np.sign(aligned) == np.sign(target)))
    rows.append({"sigma2": float(s2), "spearman": rho, "mae": mae, "sign_agree": sign_agree})

scan = pd.DataFrame(rows).sort_values("spearman", ascending=False)
best = scan.iloc[0]
print(scan.head(8).to_string(index=False))
print("\nBest rank match: σ² =", best.sigma2, "spearman =", round(best.spearman, 4), "sign_agree =", round(best.sign_agree, 3))

check(
    "recreated soft AI−nonAI ranks match saved table (Spearman ≥ 0.9)",
    best.spearman >= 0.9,
    f"spearman={best.spearman:.3f} at sigma2={best.sigma2:.4g}",
)
check(
    "sign agreement ≥ 0.8 at best σ²",
    best.sign_agree >= 0.8,
    f"sign_agree={best.sign_agree:.2f}",
)

# Working weights at best-matching σ² for later cells
W_star, lens_names = ms.compute_mixture_weights(centroids, mapping, float(best.sigma2))
recreated = doc_weighted_ai_minus_nonai(W_star, lens_names, phrases).reindex(saved_gaps.lens)
compare = saved_gaps.assign(recreated=recreated.to_numpy(), abs_err=(recreated.to_numpy() - saved_gaps.ai_minus_doc_weighted).abs())
display(compare.round(4))


## Soft group compare (CIP-0007)

`soft_group_compare` is document-bootstrap over per-doc mean mixture rows. Map AI / non-AI into a binary `group` column and confirm it returns a two-group table with difference columns. (Pub's saved table used phrase-mean pooling, not doc bootstrap — so we check API shape and direction, not bit-exact equality.)


In [ ]:
phrases_g = phrases.copy()
phrases_g["group"] = np.where(phrases_g.technology_meta == "AI", "AI", "nonAI")
phrases_g = phrases_g.rename(columns={"cluster_id": "cluster"})
# soft_group_compare expects doc_id — use source_file+chunk as doc proxy if needed
if "doc_id" not in phrases_g.columns:
    phrases_g["doc_id"] = phrases_g["source_file"].astype(str)

rng = np.random.default_rng(0)
sg = soft_group_compare(
    phrases_g,
    W_star,
    list(lens_names),
    "group",
    rng,
    groups=["nonAI", "AI"],
    cluster_col="cluster",
    reps=200,
    min_gap=0.5,  # shares are on [0,1] scale ×100 inside helper
)
display(sg.head(12))

# Direction vs saved: AI_pct - nonAI_pct should rank-correlate with saved gaps
diff_col = "difference"  # AI minus nonAI when groups=["nonAI","AI"]
sg_aligned = sg.set_index("lens").reindex(saved_gaps.lens)
rho_sg = float(spearmanr(sg_aligned[diff_col], saved_gaps.ai_minus_doc_weighted * 100).correlation)
check(
    "soft_group_compare AI−nonAI ranks agree with saved gaps",
    rho_sg >= 0.85,
    f"spearman={rho_sg:.3f}",
)


## σ² neighbour rules (CIP-0006)

Without stored multi-run lens maps we **perturb** the canonical mapping to build three synthetic runs, run `sigma2_sweep_select`, then verify neighbours are adjacent grid points of constrained σ²*.


In [ ]:
def perturb_mapping(mapping: dict, rng: np.random.Generator, frac: float = 0.15) -> dict:
    """Reassign a random fraction of clusters to a different lens."""
    lenses = list(mapping)
    # invert
    owner = {}
    for L, info in mapping.items():
        for c in info["cluster_ids"]:
            owner[int(c)] = L
    clusters = list(owner)
    n_swap = max(1, int(frac * len(clusters)))
    for c in rng.choice(clusters, size=n_swap, replace=False):
        others = [L for L in lenses if L != owner[c]]
        owner[c] = others[int(rng.integers(0, len(others)))]
    out = {L: {"cluster_ids": [], "description": mapping[L].get("description", "")} for L in lenses}
    for c, L in owner.items():
        out[L]["cluster_ids"].append(c)
    for L in out:
        out[L]["cluster_ids"] = sorted(out[L]["cluster_ids"])
    return out


rng_m = np.random.default_rng(1)
mappings_list = [mapping] + [perturb_mapping(mapping, rng_m) for _ in range(2)]
# Smaller grid for speed on 75×3072 centroids
sigma_grid = np.logspace(-4, 1, 17)
sweep = ms.sigma2_sweep_select(centroids, mappings_list, sigma2_grid=sigma_grid)
curve = pd.DataFrame(sweep["curve"])
n_lens = float(ms.mean_n_lenses(mappings_list))
lo, hi = sweep["low_cap"], sweep["high_cap"]
curve["in_band"] = (curve.mean_eff_support > lo) & (curve.mean_eff_support < hi)
print("synthetic-run σ²* =", sweep["sigma2_best"], "selection =", sweep["selection"], "n_lenses≈", n_lens)
display(curve[["sigma2", "mean_eff_support", "mean_emd", "in_band"]].round(4).head(12))

constrained = ms.select_sigma2_constrained(curve, n_lens)
harder = ms.select_sigma2_one_step_harder(curve, n_lens)
softer = ms.select_sigma2_one_step_softer(curve, n_lens)
hard = ms.select_sigma2_hard(curve, n_lens)
grid_sorted = np.sort(curve.sigma2.to_numpy())
i = int(np.argmin(np.abs(grid_sorted - constrained)))

check("constrained equals sweep sigma2_best", np.isclose(constrained, sweep["sigma2_best"]))
check("one_step_harder is previous grid point (or same if at edge)",
      np.isclose(harder, grid_sorted[max(i - 1, 0)]))
check("one_step_softer is next grid point (or same if at edge)",
      np.isclose(softer, grid_sorted[min(i + 1, len(grid_sorted) - 1)]))
check("hard is grid minimum", np.isclose(hard, grid_sorted[0]))
check("apply_sigma2_rule('constrained') matches",
      np.isclose(ms.apply_sigma2_rule("constrained", curve, n_lens), constrained))


## Document distance (CIP-0005 distance)

Build per-document cluster distributions from phrases and run EMD + PERMANOVA for AI vs non-AI (subset for speed). Confirms `multilogue.distance` owns this path — study notebooks only supply the group column.


In [ ]:
# Subsample documents for a quick distance smoke (full matrix is expensive).
doc_col = "source_file"
cluster_col = "cluster_id"
# one row per (doc, cluster) count
sub_docs = (
    phrases.groupby(doc_col)["technology_meta"]
    .agg(lambda s: "AI" if (s == "AI").mean() > 0.5 else "nonAI")
    .rename("group")
)
# take up to 40 docs per group
rng_d = np.random.default_rng(0)
picked = []
for g, idx in sub_docs.groupby(sub_docs):
    choose = list(idx.index)
    picked.extend(rng_d.choice(choose, size=min(40, len(choose)), replace=False).tolist())
phrases_d = phrases[phrases[doc_col].isin(picked)].copy()
groups = sub_docs.loc[sorted(set(picked))]

P, doc_ids, _totals = dd.document_cluster_distributions(
    phrases_d.rename(columns={doc_col: "doc_id", cluster_col: "cluster"}),
    n_clusters=centroids.shape[0],
    doc_col="doc_id",
    cluster_col="cluster",
    min_phrases=5,  # smoke: keep short docs; production uses a higher floor
)
g = groups.reindex(doc_ids).to_numpy()
M = dd.cosine_ground_cost(centroids)
D = dd.emd_distance_matrix(P, M)
perm = dd.permanova(D, g, n_perm=99, seed=0)
print("PERMANOVA AI vs nonAI (smoke):", {k: (perm[k] if k in perm else perm.get(k)) for k in ("pseudo_F", "p_value", "n")})
check("document EMD matrix finite", np.isfinite(D).all(), f"shape={D.shape}")
check("PERMANOVA returned pseudo_F", np.isfinite(float(perm["pseudo_F"])))


## Keep-local reminders (not imported from multilogue)

These stay in `pub_dialogue` / study notebooks — the gap analysis said **do not** promote them:

1. **Legacy σ² rules** `min_emd`, `median` (`LEGACY_SIGMA2_RULES` in `multiverse.py`)  
2. **Headline ℓ** — `ell_R1_*`, `ell_R2_*`, `ell_R3_*` (tech coverage, AI salience ranking, temporal entropy)  
3. **Access / assess** — PDF chunking, bibliography heuristics  
4. **Extraction prompts** and technology-term filters  

Confirm legacy still exists locally and is separate from multilogue:


In [ ]:
sys.path.insert(0, str(ROOT))
from pub_dialogue import multiverse as mv

check(
    "pub still exposes LEGACY_SIGMA2_RULES locally",
    hasattr(mv, "LEGACY_SIGMA2_RULES") and "min_emd" in mv.LEGACY_SIGMA2_RULES,
    str(list(getattr(mv, "LEGACY_SIGMA2_RULES", {}))),
)
check(
    "pub SIGMA2_RULES is multilogue's (thin re-export)",
    list(mv.SIGMA2_RULES) == list(ms.SIGMA2_RULES),
)
check(
    "ell_R2 stays in pub (study metric)",
    callable(getattr(mv, "ell_R2_max_pp_gap", None)),
)


## Summary


In [ ]:
summary = pd.DataFrame(checks)
display(summary)
n_fail = (~summary["pass"]).sum()
print(f"\n{len(summary) - n_fail}/{len(summary)} checks passed.")
if n_fail:
    print("FAILURES:\n", summary.loc[~summary["pass"]].to_string(index=False))
    raise SystemExit(f"{n_fail} gap/recreation checks failed")
print("All gap + recreation checks passed.")
